In [1]:
# Python version 3.11.7

# Load general use packages
import pandas as pd
import numpy as np
import os

# Load torch for making tensors
import torch

# Local scripts
from multi_mlp import simple_FC, JointMLP

## Define Inputs

Here, we will define one model per omic. Hidden layers can be set and should account for data size. DeepIMV cannot handle missing values, so a dataset with missing values removed and/or imputed is needed. 

In [2]:
# Load datasets
metabolomics = pd.read_csv("../../Dataset/Scaled/Metabolomics.txt", delimiter = "\t").drop("Feature", axis = 1)
print("Metabolomics has " + str(len(metabolomics)) + " features")

lippos = pd.read_csv("../../Dataset/Scaled/Lipidomics_Positive.txt", delimiter = "\t").drop("Feature", axis = 1)
print("Lipidomics Positive has " + str(len(lippos)) + " features")

lipneg = pd.read_csv("../../Dataset/Scaled/Lipidomics_Negative.txt", delimiter = "\t").drop("Feature", axis = 1)
print("Lipidomics Negative has " + str(len(lipneg)) + " features")

Metabolomics has 54 features
Lipidomics Positive has 81 features
Lipidomics Negative has 34 features


## Hyperparameter Tuning

| Approximate Proportional Size of Datasets | Root of 2 Size: Lip Neg | Root of 2 Size: Lip Pos + Metabolomics |
|---|---|---|
| 1/4 | 8 | 16 |
| 1/2 | 16 | 32 |
| 1 | 32 | 64 |
| 2 | 64 | 128 |
| 4 | 128 | 256 |

In [3]:
fdata = pd.read_csv("../../Dataset/Scaled/FData.csv")

# Load splits 
splits = []
truths = []
with open("splits.txt", "r") as file:
    for line in file:
        values = line.replace("\n", "").split(" ")
        values = [int(x)-1 for x in values] # Correct for indexing changes between R and python
        splits.append(values)
        truths.append([0 if "N" in el else 1 for el in fdata["Group"][values]])

print(splits)
print(truths)

[[0, 1, 2, 3, 5, 8, 9, 10, 11, 12, 14, 16, 17, 20, 21, 25, 27, 29, 30, 31, 32, 33, 0], [0, 2, 3, 4, 6, 7, 8, 9, 10, 12, 13, 15, 18, 19, 20, 22, 23, 24, 25, 26, 28, 29, 32], [1, 4, 5, 6, 7, 11, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 26, 27, 28, 30, 31, 33], [3, 5, 6, 7, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 23, 24, 25, 26, 28, 29, 30, 31, 3], [0, 1, 2, 4, 5, 7, 8, 9, 10, 15, 16, 17, 18, 19, 20, 21, 22, 26, 27, 28, 30, 32, 33], [0, 1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 13, 14, 17, 22, 23, 24, 25, 27, 29, 31, 32, 33], [0, 2, 3, 4, 6, 8, 9, 13, 14, 15, 16, 17, 18, 20, 21, 27, 28, 29, 30, 31, 32, 33, 0], [0, 1, 3, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 19, 20, 22, 23, 24, 25, 26, 28, 30, 33], [1, 2, 4, 5, 7, 8, 10, 11, 12, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 29, 31, 32], [0, 3, 4, 6, 9, 10, 11, 12, 13, 15, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 0], [1, 2, 4, 5, 7, 8, 9, 10, 12, 14, 16, 17, 18, 19, 21, 23, 24, 26, 27, 29, 31, 32, 33], [0, 1, 2, 3, 5, 6, 7, 8, 11, 13, 14, 15, 

In [4]:
# Define sizes
small_size = [8, 16, 32, 64, 128]
large_size = [16, 32, 64, 128, 256]

# Hold all final loss values
final_loss = []

# Iterate through sizes 
for el in range(len(small_size)):

    # Iterate through splits
    for el2 in range(len(splits)):

        print("Element Size Test:", el, "and Crossfold Number:", el2)

        # Define groups (thankfully always 2 00-week and 6 post-00 week)
        groups = torch.tensor(truths[el])

        # Define models 
        metab_train = simple_FC(input_size = metabolomics.shape[0], hidden_sizes = [large_size[el], 64], prediction_dim = 2, dropout = 0.5)
        lip_pos_train = simple_FC(input_size = lippos.shape[0], hidden_sizes = [large_size[el], 64], prediction_dim = 2, dropout = 0.5)
        lip_neg_train = simple_FC(input_size = lipneg.shape[0], hidden_sizes = [large_size[el], 64], prediction_dim = 2, dropout = 0.5)

        # Define joint model
        joint_mlp = JointMLP(marginal_models = [metab_train, lip_pos_train, lip_neg_train], hidden_dim = 64, hooks=False)

        # Optimize parameters 
        optimizer_mlp = torch.optim.AdamW(joint_mlp.parameters(), lr=1e-4)

        # Define data views with subsetting
        views = [torch.tensor(metabolomics.iloc[:,splits[el2]].T.values, dtype = torch.float32),
                torch.tensor(lippos.iloc[:,splits[el2]].T.values, dtype = torch.float32),
                torch.tensor(lipneg.iloc[:,splits[el2]].T.values, dtype = torch.float32)]

        acc_mlp = []

        # Time: 20 seconds             
        for i in range(1500):

            # Update the mlp
            yhat, h, yhats, hiddens = joint_mlp(*views)

            # pass the predictions and distributions to the loss function and update parameters
            _, _, loss = joint_mlp.loss(groups, yhat, yhats)

            optimizer_mlp.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(joint_mlp.parameters(), 2.0)
            optimizer_mlp.step()

        final_loss.append(loss.item())

Element Size Test: 0 and Crossfold Number: 0
Element Size Test: 0 and Crossfold Number: 1
Element Size Test: 0 and Crossfold Number: 2
Element Size Test: 0 and Crossfold Number: 3
Element Size Test: 0 and Crossfold Number: 4
Element Size Test: 0 and Crossfold Number: 5
Element Size Test: 0 and Crossfold Number: 6
Element Size Test: 0 and Crossfold Number: 7
Element Size Test: 0 and Crossfold Number: 8
Element Size Test: 0 and Crossfold Number: 9
Element Size Test: 0 and Crossfold Number: 10
Element Size Test: 0 and Crossfold Number: 11
Element Size Test: 0 and Crossfold Number: 12
Element Size Test: 0 and Crossfold Number: 13
Element Size Test: 0 and Crossfold Number: 14
Element Size Test: 1 and Crossfold Number: 0
Element Size Test: 1 and Crossfold Number: 1
Element Size Test: 1 and Crossfold Number: 2
Element Size Test: 1 and Crossfold Number: 3
Element Size Test: 1 and Crossfold Number: 4
Element Size Test: 1 and Crossfold Number: 5
Element Size Test: 1 and Crossfold Number: 6
Eleme

In [6]:
sizes = [0.25 for x in range(15)]
sizes.extend([0.5 for x in range(15)])
sizes.extend([1 for x in range(15)])
sizes.extend([2 for x in range(15)])
sizes.extend([4 for x in range(15)])

pd.DataFrame({
    "Size": sizes,
    "Loss": final_loss
}).to_csv("MultiMLP_tuning.csv", index = False)

## Define and Run Models

In [7]:
# Define all four models. Predicition dim is the number of categories.
metab_model = simple_FC(input_size = metabolomics.shape[0], hidden_sizes = [128, 64], prediction_dim = 2, dropout = 0.5)
lip_pos_model = simple_FC(input_size = lippos.shape[0], hidden_sizes = [128, 64], prediction_dim = 2, dropout = 0.5)
lip_neg_model = simple_FC(input_size = lipneg.shape[0], hidden_sizes = [64, 64], prediction_dim = 2, dropout = 0.5)

In [8]:
# Define joint model
joint_mlp = JointMLP(marginal_models = [metab_model, lip_pos_model, lip_neg_model], hidden_dim = 64, hooks=False)

In [9]:
# Optimize parameters 
optimizer_mlp = torch.optim.AdamW(joint_mlp.parameters(), lr=1e-4)

In [12]:
groups = torch.tensor([0,0,0,0,0,0,1,0,1,1,0,0,1,1,1,1,0,1,0,1,1,1,1,0,1,1,0,1,1,0,1,0,0,1])

# Define data views
views = [torch.tensor(metabolomics.T.values, dtype = torch.float32),
         torch.tensor(lippos.T.values, dtype = torch.float32),
         torch.tensor(lipneg.T.values, dtype = torch.float32)]

acc_mlp = []

# Time: 20 seconds             
for i in range(1500):

    # Update the mlp
    yhat, h, yhats, hiddens = joint_mlp(*views)

    # pass the predictions and distributions to the loss function and update parameters
    _, _, loss = joint_mlp.loss(groups, yhat, yhats)

    optimizer_mlp.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(joint_mlp.parameters(), 2.0)
    optimizer_mlp.step()

    print(f'Epoch {i+1} loss: {loss.item():.3f}')

Epoch 1 loss: 0.395
Epoch 2 loss: 0.355
Epoch 3 loss: 0.381
Epoch 4 loss: 0.348
Epoch 5 loss: 0.365
Epoch 6 loss: 0.363
Epoch 7 loss: 0.383
Epoch 8 loss: 0.365
Epoch 9 loss: 0.377
Epoch 10 loss: 0.362
Epoch 11 loss: 0.368
Epoch 12 loss: 0.348
Epoch 13 loss: 0.356
Epoch 14 loss: 0.374
Epoch 15 loss: 0.365
Epoch 16 loss: 0.369
Epoch 17 loss: 0.356
Epoch 18 loss: 0.352
Epoch 19 loss: 0.361
Epoch 20 loss: 0.366
Epoch 21 loss: 0.356
Epoch 22 loss: 0.355
Epoch 23 loss: 0.358
Epoch 24 loss: 0.381
Epoch 25 loss: 0.366
Epoch 26 loss: 0.370
Epoch 27 loss: 0.363
Epoch 28 loss: 0.360
Epoch 29 loss: 0.354
Epoch 30 loss: 0.364
Epoch 31 loss: 0.340
Epoch 32 loss: 0.373
Epoch 33 loss: 0.345
Epoch 34 loss: 0.359
Epoch 35 loss: 0.358
Epoch 36 loss: 0.361
Epoch 37 loss: 0.354
Epoch 38 loss: 0.370
Epoch 39 loss: 0.351
Epoch 40 loss: 0.357
Epoch 41 loss: 0.368
Epoch 42 loss: 0.347
Epoch 43 loss: 0.367
Epoch 44 loss: 0.364
Epoch 45 loss: 0.350
Epoch 46 loss: 0.362
Epoch 47 loss: 0.349
Epoch 48 loss: 0.351
E

In [13]:
joint_mlp.eval()

JointMLP(
  (margin_models): ModuleList(
    (0): simple_FC(
      (fc1): Linear(in_features=54, out_features=128, bias=True)
      (fc2): Linear(in_features=128, out_features=64, bias=True)
      (fc_out): Linear(in_features=64, out_features=2, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
    )
    (1): simple_FC(
      (fc1): Linear(in_features=81, out_features=128, bias=True)
      (fc2): Linear(in_features=128, out_features=64, bias=True)
      (fc_out): Linear(in_features=64, out_features=2, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
    )
    (2): simple_FC(
      (fc1): Linear(in_features=34, out_features=64, bias=True)
      (fc2): Linear(in_features=64, out_features=64, bias=True)
      (fc_out): Linear(in_features=64, out_features=2, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
    )
  )
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
  (dropout): Dropout(p=0.2, inp

In [14]:
for name, param in joint_mlp.named_parameters():
    if 'weight' in name:
        print(f"Layer: {name}, Size: {param.data.shape}")

Layer: margin_models.0.fc1.weight, Size: torch.Size([128, 54])
Layer: margin_models.0.fc2.weight, Size: torch.Size([64, 128])
Layer: margin_models.0.fc_out.weight, Size: torch.Size([2, 64])
Layer: margin_models.1.fc1.weight, Size: torch.Size([128, 81])
Layer: margin_models.1.fc2.weight, Size: torch.Size([64, 128])
Layer: margin_models.1.fc_out.weight, Size: torch.Size([2, 64])
Layer: margin_models.2.fc1.weight, Size: torch.Size([64, 34])
Layer: margin_models.2.fc2.weight, Size: torch.Size([64, 64])
Layer: margin_models.2.fc_out.weight, Size: torch.Size([2, 64])
Layer: fc1.weight, Size: torch.Size([64, 64])
Layer: fc2.weight, Size: torch.Size([2, 64])


In [15]:
# V Matrix (V1)
joint_mlp.margin_models[0].fc1.weight

Parameter containing:
tensor([[-0.1319, -0.0437,  0.0565,  ..., -0.0607, -0.0924,  0.0770],
        [ 0.1567, -0.1025, -0.1406,  ...,  0.0412, -0.0549,  0.0261],
        [-0.0404,  0.0510, -0.0253,  ...,  0.1440,  0.1351, -0.1271],
        ...,
        [ 0.0596, -0.1021,  0.0377,  ...,  0.0692,  0.0775, -0.1085],
        [-0.1205,  0.0524, -0.1187,  ...,  0.0370, -0.0621,  0.0041],
        [-0.0257,  0.0670,  0.1324,  ...,  0.0366,  0.0839,  0.0390]],
       requires_grad=True)

In [16]:
# Weights for Features in V1 (V2)
joint_mlp.margin_models[0].fc2.weight

Parameter containing:
tensor([[ 0.0619, -0.0119, -0.0654,  ..., -0.0048,  0.0518,  0.1034],
        [-0.0689, -0.0037, -0.0496,  ...,  0.0094, -0.0467,  0.0312],
        [ 0.0656,  0.0945,  0.1299,  ..., -0.0186,  0.0419, -0.0687],
        ...,
        [ 0.0237, -0.0849, -0.1099,  ..., -0.0211, -0.0397,  0.1049],
        [-0.0312, -0.1009, -0.1049,  ..., -0.0263, -0.0304,  0.0942],
        [ 0.0183, -0.0125, -0.0677,  ..., -0.0418,  0.0164,  0.0894]],
       requires_grad=True)

In [17]:
# Each V2 weight for its class [0 or 1] (V3)
joint_mlp.margin_models[0].fc_out.weight

Parameter containing:
tensor([[-0.0921, -0.0935,  0.0925,  0.0691,  0.1068, -0.0516, -0.0912,  0.0820,
         -0.1347, -0.1039, -0.0275,  0.1021, -0.0986,  0.0056, -0.0462, -0.1144,
         -0.1271,  0.0822, -0.0708, -0.1352,  0.0544,  0.1590,  0.0893, -0.1364,
         -0.0632,  0.0393,  0.0595,  0.0474,  0.0092, -0.0661,  0.0984, -0.0408,
         -0.1053,  0.0318,  0.0227,  0.1203, -0.1029, -0.0936,  0.0067, -0.0221,
         -0.1235,  0.0386,  0.1011, -0.1313,  0.0394,  0.1150,  0.1283,  0.0337,
          0.0414, -0.1060,  0.0245, -0.1016,  0.1131, -0.1378, -0.0995,  0.1267,
          0.0928,  0.0711,  0.1372, -0.1290,  0.0582, -0.0362, -0.0229, -0.1085],
        [ 0.0578,  0.0423, -0.0826, -0.1062, -0.0444,  0.1431,  0.1138, -0.1074,
          0.0910, -0.0948, -0.1087, -0.0230,  0.0449, -0.1483,  0.0058, -0.0363,
         -0.0177, -0.0719, -0.0611,  0.1197, -0.0340,  0.0724, -0.0363,  0.0010,
          0.0955, -0.0822, -0.0081,  0.1170, -0.0994, -0.0681, -0.0652,  0.1359,
     

In [18]:
joint_mlp.margin_models[2].fc_out.weight

Parameter containing:
tensor([[-7.9976e-02,  1.6576e-01,  5.2896e-02, -3.1293e-02, -6.3700e-02,
         -1.0396e-01,  4.8270e-02, -1.2331e-01, -4.6645e-02,  9.8690e-02,
          1.2921e-01, -1.3134e-01, -1.1611e-01, -1.0651e-01, -1.3555e-01,
          5.3974e-02,  4.7204e-03,  8.8515e-02,  8.6301e-03, -8.3411e-02,
          7.6518e-02, -7.4590e-02, -2.0288e-02, -6.1501e-02,  2.1319e-02,
          5.5620e-02,  5.1945e-02,  1.0479e-01, -6.8120e-03, -4.5106e-02,
          2.2060e-02,  1.1190e-01, -1.3807e-01, -5.4327e-02, -1.7807e-01,
         -6.5490e-03, -1.5396e-01, -8.2733e-02, -5.5811e-02,  7.6104e-02,
          1.3380e-02,  8.0845e-02,  8.7020e-02,  9.7752e-02,  4.9699e-02,
         -3.7860e-02, -1.1616e-01, -5.0509e-02,  3.4481e-02, -7.2115e-02,
          1.1623e-01,  2.3335e-02,  1.1388e-01, -1.2396e-01,  2.6305e-02,
         -6.3046e-02, -9.5453e-02,  3.0657e-02,  1.1187e-01, -2.2574e-02,
          1.1434e-01,  3.6296e-02, -1.1852e-01, -5.5770e-02],
        [-2.8302e-02, -1.459

In [19]:
import shap

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [20]:
import torch.nn as nn

# need to wrap the model in this class to get around some issues with the SHAP package
class JointMLPWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
 
    def forward(self, *datas):
        yhat, _, _, _ = self.model(*datas)
        return yhat
    
joint_model_wrp = JointMLPWrapper(joint_mlp)

# make the explainer object
explainer = shap.DeepExplainer(joint_model_wrp, views)

In [21]:
shap_values = explainer.shap_values(views, check_additivity=False)

In [ ]:
# First index is which class (00wk or post-00wk). The second index is which view. The third index is the shapley values by samples per feature.
# Shapley values were summed across samples. 

shap_vals = pd.concat([
    pd.DataFrame({
        "Shapley Value": pd.DataFrame(shap_values[0][0]).sum().tolist(),
        "Feature": pd.read_csv("../../Dataset/Scaled/Metabolomics.txt", sep = "\t")["Feature"].tolist(),
        "View": ["metabolomics" for x in range(len(metabolomics))]
    }),
    pd.DataFrame({
        "Shapley Value": pd.DataFrame(shap_values[0][1]).sum().tolist(),
        "Feature": pd.read_csv("../../Dataset/Scaled/Lipidomics_Positive.txt", sep = "\t")["Feature"].tolist(),
        "View": ["lipidomics positive" for x in range(len(lippos))]
    }),
    pd.DataFrame({
        "Shapley Value": pd.DataFrame(shap_values[0][2]).sum().tolist(),
        "Feature": pd.read_csv("../../Dataset/Scaled/Lipidomics_Negative.txt", sep = "\t")["Feature"].tolist(),
        "View": ["lipidomics negative" for x in range(len(lipneg))]
    }),
]).reset_index(drop = True)

shap_vals.to_csv("MultiMLP_ShapValues.csv", index = False)

shap_vals


,Shapley Value,Feature,View
0,-0.005372,(+)-6-aminopenicillanic acid,metabolomics
1,0.000449,"1,3-dihydroxyacetone",metabolomics
2,-0.001708,1-methyladenosine,metabolomics
3,-0.001832,2-methyl-3-hydroxybutyric acid,metabolomics
4,0.000938,3-(1-pyrazolyl)-L-alanine,metabolomics
...,...,...,...
164,0.000216,PE 33:2,lipidomics negative
165,-0.000126,PG 32:0,lipidomics negative
166,-0.002714,FA 41:0,lipidomics negative
167,0.000071,HexCer 49:6;2O,lipidomics negative
